# Fine-tune PP-OCRv5 mobile — 2 epoch

Notebook này fine-tune mô hình nhận diện chữ `PP-OCRv5_mobile_rec` trên tập huấn luyện `rec_train.txt` (93.997 dòng) trong 2 epoch với từ điển tiếng Việt `vi_dict.txt`, rồi đánh giá checkpoint cuối trên `rec_test.txt` bằng đúng `src/metrics.py` đã dùng cho baseline. Hai notebook fine-tune (2 và 4 epoch) có code giống hệt nhau và dùng hai config chỉ khác `Global.epoch_num` cùng đường dẫn lưu kết quả, nên chênh lệch kết quả chỉ do số epoch. Notebook chạy trên Kaggle với GPU T4 x2 và Internet; nếu có Kaggle Secret `huggingface.co` (quyền ghi) thì checkpoint được đẩy lên Hugging Face ở cell cuối, nếu không thì bước đó tự bỏ qua.

## 1. Cài đặt môi trường

Cài PaddlePaddle GPU 3.2.1 (CUDA 12.6), PaddleOCR 3.3.0 và PaddleX 3.3.0 với phiên bản cố định, để baseline và cả hai lần fine-tune chạy trong cùng một môi trường và kết quả tái lập được. File cài PaddlePaddle nặng khoảng 1,6 GB nên được tải riêng rồi cài từ file, tránh bị đứt khi tải thẳng. Cell cũng gỡ `torch` và `langchain` có sẵn của Kaggle vì xung đột với Paddle và không được notebook dùng, sau đó kiểm tra lại phiên bản đã cài; nếu bước này lỗi thì các bước sau không có ý nghĩa.

In [ ]:
import subprocess
import sys
import zipfile
from importlib.metadata import version
from pathlib import Path

PADDLE_VERSION = "3.2.1"
PADDLEOCR_VERSION = "3.3.0"
PADDLEX_VERSION = "3.3.0"  # PaddleOCR 3.3.0 yêu cầu paddlex>=3.3.0,<3.4.0
CUDA_TAG = "cu126"  # driver Kaggle (CUDA 12.x trở lên) chạy được bản build này

PADDLE_INDEX = f"https://www.paddlepaddle.org.cn/packages/stable/{CUDA_TAG}/"
WHEEL_DIR = Path("/kaggle/temp")  # thư mục tạm, không bị lưu vào Output
WHEEL_DIR.mkdir(parents=True, exist_ok=True)

# Dọn wheel cũ nếu từng tải nhầm vào /kaggle/working (làm phình Output)
for stale in Path("/kaggle/working").glob("paddlepaddle_gpu-*.whl"):
    stale.unlink()

subprocess.run(["nvidia-smi"], check=True)

# 1) Tải riêng file wheel bằng pip download (pip tự chọn đúng tên file theo phiên bản Python), kiểm tra file nguyên vẹn
for attempt in range(1, 4):
    for old in WHEEL_DIR.glob("paddlepaddle_gpu-*.whl"):
        old.unlink()
    subprocess.run(
        [sys.executable, "-m", "pip", "download", f"paddlepaddle-gpu=={PADDLE_VERSION}",
         "--no-deps", "-d", str(WHEEL_DIR), "-i", PADDLE_INDEX, "--timeout", "600", "--retries", "10"],
        check=True,
    )
    WHEEL_PATH = next(WHEEL_DIR.glob("paddlepaddle_gpu-*.whl"))
    try:
        with zipfile.ZipFile(WHEEL_PATH):  # file bị cắt cụt sẽ báo BadZipFile
            pass
        print(f"Wheel OK: {WHEEL_PATH} ({WHEEL_PATH.stat().st_size / 1e9:.2f} GB)")
        break
    except zipfile.BadZipFile:
        print(f"Lần {attempt}: wheel hỏng, xóa và tải lại")
        WHEEL_PATH.unlink(missing_ok=True)
else:
    raise RuntimeError("Không tải được wheel nguyên vẹn sau 3 lần.")

# 2) Cài Paddle từ file local (dependency lấy từ PyPI), rồi PaddleOCR, PaddleX và thư viện phụ
pip = [sys.executable, "-m", "pip", "install", "--timeout", "600", "--retries", "10"]
subprocess.run(pip + [str(WHEEL_PATH)], check=True)
subprocess.run(
    pip + [
        f"paddleocr=={PADDLEOCR_VERSION}",
        f"paddlex=={PADDLEX_VERSION}",
        "python-Levenshtein>=0.25",
        "PyYAML>=6.0",
        "tqdm>=4.0",
    ],
    check=True,
)

# 2b) Gỡ các gói có sẵn của Kaggle mà notebook này không dùng nhưng gây lỗi:
#  - torch: Paddle hạ cấp nvidia-nccl/cudnn làm torch hỏng
#  - langchain: bản mới của Kaggle (1.x) không còn module langchain.docstore, làm `import paddleocr` báo ModuleNotFoundError
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y",
     "torch", "torchvision", "torchaudio", "langchain", "langchain-community"],
    check=False,
)

# 3) Xác nhận: đúng phiên bản đã ghim, và Paddle chạy được trên GPU (tiến trình mới nên không cần restart)
subprocess.run(
    [sys.executable, "-m", "pip", "show", "paddlepaddle-gpu", "paddleocr", "paddlex"],
    check=True,
)
for package, expected in [
    ("paddlepaddle-gpu", PADDLE_VERSION),
    ("paddleocr", PADDLEOCR_VERSION),
    ("paddlex", PADDLEX_VERSION),
]:
    installed = version(package)
    assert installed == expected, f"{package}: cài {installed}, cần {expected}"
subprocess.run(
    [sys.executable, "-c",
     "import paddle; paddle.utils.run_check(); print(paddle.__version__, paddle.device.get_device())"],
    check=True,
)

## 2. Ghi nhận môi trường chạy và cố định seed

Đặt seed 42 để giảm tính ngẫu nhiên giữa các lần chạy, kiểm tra PaddlePaddle nhận được GPU và ghi lại phiên bản Python, các thư viện và GPU. Thông tin này được lưu cùng kết quả để báo cáo nêu rõ điều kiện đo. Nếu cell báo lỗi import ngay sau khi vừa cài ở mục 1, cần khởi động lại session rồi chạy lại từ mục 2.

In [ ]:
import json
import os
import platform
import random
import subprocess
import sys
import time
import warnings
from importlib.metadata import version
from pathlib import Path

warnings.filterwarnings("ignore", message="No ccache found")  # cảnh báo vô hại của Paddle

# Đặt trước khi tạo model. BOS là nguồn model chính thức dự phòng khi Hugging Face không truy cập được.
os.environ["PADDLE_PDX_MODEL_SOURCE"] = "BOS"

import numpy as np
import paddle
import paddleocr

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
paddle.seed(SEED)

assert paddle.is_compiled_with_cuda(), (
    "PaddlePaddle has no CUDA support. Enable a Kaggle GPU and rerun installation."
)

gpu_info = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
    text=True,
).strip()

ENV_INFO = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "paddlepaddle": paddle.__version__,
    "paddleocr": paddleocr.__version__,
    "paddlex": version("paddlex"),
    "seed": SEED,
    "gpu": gpu_info.splitlines(),
}
print(json.dumps(ENV_INFO, ensure_ascii=False, indent=2))

## 3. Lấy code dùng chung từ repository

Hàm tính metric nằm trong `src/metrics.py` của repository và được dùng cho mọi lần đánh giá (baseline và fine-tune), nên notebook clone repository về `/kaggle/working/repo` và import trực tiếp thay vì chép lại. Mã commit được ghi lại để biết mỗi con số ứng với phiên bản code nào.

In [ ]:
REPO_URL = "https://github.com/nguyenngocbao/nlp-vietnamese-ocr-recognition.git"
REPO_REF = "main"  # có thể đổi thành mã commit hoặc tag để cố định hẳn phiên bản code
REPO_DIR = Path("/kaggle/working/repo")

if not (REPO_DIR / "src" / "metrics.py").is_file():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--quiet", REPO_REF], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

ENV_INFO["repo_url"] = REPO_URL
ENV_INFO["repo_commit"] = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
print("Repo commit:", ENV_INFO["repo_commit"])

## 4. Tìm dữ liệu

Kaggle gắn dữ liệu ở thư mục chỉ đọc `/kaggle/input`, đã giải nén sẵn gồm thư mục `vi_rec_100k/` (ba file nhãn train, validation, test cùng ảnh) và từ điển `vi_dict.txt`. Cell xác định đường dẫn thực tế của các file này (thử cả hai kiểu đường dẫn mà Kaggle dùng) để các bước sau không phải viết cố định đường dẫn.

In [ ]:
DATASET_SLUG = "hcmus-nlp-vietnamese-recognition-resources"
DATASET_CANDIDATES = [
    Path("/kaggle/input/datasets/baonguyenngoc0902") / DATASET_SLUG,  # kiểu mount hiện tại
    Path("/kaggle/input") / DATASET_SLUG,  # kiểu mount cũ
]
DATASET_DIR = next((path for path in DATASET_CANDIDATES if path.exists()), None)
assert DATASET_DIR is not None, (
    "Attach the Kaggle dataset in the Input panel. Đã thử: " + ", ".join(map(str, DATASET_CANDIDATES))
)

WORK_DIR = Path("/kaggle/working")
DATA_DIR = DATASET_DIR  # dữ liệu đã giải nén sẵn, đọc thẳng từ Input (chỉ đọc)
RESULTS_DIR = WORK_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def find_file(name: str) -> Path:
    # Chỉ kiểm tra hai vị trí đã biết. Không dùng rglob vì thư mục chứa hàng trăm nghìn ảnh, quét rất chậm trên ổ Input
    candidates = [DATASET_DIR / name, DATASET_DIR / "vi_rec_100k" / name]
    matches = [path for path in candidates if path.is_file()]
    assert matches, f"Không thấy {name} trong: {candidates}"
    return matches[0]  # vị trí nông nhất được ưu tiên

DICT_PATH = find_file("vi_dict.txt")
TRAIN_LABELS = find_file("rec_train.txt")
VAL_LABELS = find_file("rec_val.txt")
TEST_LABELS = find_file("rec_test.txt")

print("dataset:", DATASET_DIR)
print("dict:", DICT_PATH, f"({len(DICT_PATH.read_text(encoding='utf-8').splitlines())} dòng)")
print("train labels:", TRAIN_LABELS)
print("val labels:", VAL_LABELS)
print("test labels:", TEST_LABELS)

## 5. Kiểm tra các file nhãn

Mỗi dòng nhãn gồm đường dẫn ảnh, một dấu tab và văn bản đúng. Cell đọc ba tập, kiểm tra số dòng khớp đề bài (93.997 train, 3.000 validation, 3.003 test) và in vài nhãn test đầu tiên, để bảo đảm dữ liệu đủ và đúng trước khi đo hoặc huấn luyện.

In [ ]:
def read_label_lines(path: Path) -> list[str]:
    lines = path.read_text(encoding="utf-8").splitlines()
    assert all("\t" in line for line in lines), f"Invalid label format in {path}"
    return lines

split_lines = {
    "train": read_label_lines(TRAIN_LABELS),
    "val": read_label_lines(VAL_LABELS),
    "test": read_label_lines(TEST_LABELS),
}
for split, lines in split_lines.items():
    print(f"{split}: {len(lines):,} labels")

assert len(split_lines["train"]) == 93_997
assert len(split_lines["val"]) == 3_000
assert len(split_lines["test"]) == 3_003
print("First three test labels:")
print("\n".join(split_lines["test"][:3]))


## 6. Đọc config fine-tune

Toàn bộ siêu tham số nằm trong file config của repository (`configs/ppocrv5_mobile/`), không viết tay trong notebook. Cell kiểm tra số epoch đúng với notebook này, đường dẫn dữ liệu khớp, và so với config của bản 4 epoch để xác nhận hai file chỉ khác `epoch_num`, `save_model_dir` và `save_res_path`; đây là bằng chứng cho yêu cầu mục 2.4 rằng hai cấu hình chỉ khác số epoch.

In [ ]:
import difflib
import shutil

import yaml

EPOCHS = 2
OTHER_EPOCHS = 4
RUN_NAME = f"ppocrv5_mobile_{EPOCHS}epochs"
CONFIG_DIR = REPO_DIR / "configs" / "ppocrv5_mobile"
CONFIG_PATH = CONFIG_DIR / f"ppocrv5_mobile_{EPOCHS}epochs.yml"
OTHER_PATH = CONFIG_DIR / f"ppocrv5_mobile_{OTHER_EPOCHS}epochs.yml"

config_text = CONFIG_PATH.read_text(encoding="utf-8")
config = yaml.safe_load(config_text)

assert config["Global"]["epoch_num"] == EPOCHS
assert Path(config["Global"]["character_dict_path"]) == DICT_PATH
assert Path(config["Train"]["dataset"]["data_dir"]) == TRAIN_LABELS.parent
assert Path(config["Train"]["dataset"]["label_file_list"][0]) == TRAIN_LABELS
assert Path(config["Eval"]["dataset"]["label_file_list"][0]) == VAL_LABELS
assert config["Global"]["save_epoch_step"] == 1 and config["Global"]["seed"] == SEED

diff_lines = [
    line for line in difflib.unified_diff(
        config_text.splitlines(), OTHER_PATH.read_text(encoding="utf-8").splitlines(), lineterm="", n=0)
    if line[:1] in "+-" and line[:3] not in ("+++", "---")
]
print("\n".join(diff_lines))
allowed = ("epoch_num", "save_model_dir", "save_res_path", "# Fine-tuning config")
assert all(any(key in line for key in allowed) for line in diff_lines), "Configs differ beyond epoch/output paths"

MAX_TEXT_LEN = config["Architecture"]["Head"]["head_list"][1]["NRTRHead"]["max_text_length"]
IMG_W = config["Eval"]["dataset"]["transforms"][2]["RecResizeImg"]["image_shape"][2]
print("epoch_num:", EPOCHS, "| max_text_length:", MAX_TEXT_LEN, "| image width:", IMG_W,
      "| first_bs:", config["Train"]["sampler"]["first_bs"], "| lr:", config["Optimizer"]["lr"])

## 7. Thống kê dữ liệu và kiểm tra từ điển

Hai kiểm tra trước khi tốn thời gian huấn luyện. (i) Độ dài nhãn và độ rộng ảnh: config gốc của PaddleOCR dành cho dòng ngắn (tối đa 25 ký tự, ảnh rộng 320 px), trong khi dòng tiếng Việt dài tới khoảng 80 ký tự; cell in phân vị độ dài nhãn và độ rộng ảnh để chắc `max_text_length` và chiều rộng trong config đủ lớn, vì nhãn dài hơn `max_text_length` bị PaddleOCR bỏ qua khi huấn luyện. (ii) Từ điển `vi_dict.txt`: ký tự ngoài từ điển thì mô hình không thể học, nên cell đếm tỷ lệ ký tự trong nhãn train bị thiếu trong từ điển.

In [ ]:
import random
from collections import Counter

from PIL import Image

def percentile(sorted_values, p):
    return sorted_values[min(len(sorted_values) - 1, int(len(sorted_values) * p))]

train_labels = [line.split("\t", 1)[1] for line in split_lines["train"]]
lengths = sorted(len(label) for label in train_labels)
print("Label length p50/p90/p99/max:", [percentile(lengths, p) for p in (0.5, 0.9, 0.99)], lengths[-1])
too_long = sum(n > MAX_TEXT_LEN for n in lengths)
print(f"Labels longer than max_text_length={MAX_TEXT_LEN}: {too_long} ({too_long / len(lengths):.4%})")
assert too_long / len(lengths) < 0.001, "Raise max_text_length in the config generator"

rng = random.Random(SEED)
widths = []
for line in rng.sample(split_lines["train"], 300):
    with Image.open(TRAIN_LABELS.parent / line.split("\t", 1)[0]) as image:
        widths.append(image.width * 48 / image.height)  # chiều rộng khi co về cao 48 px
widths.sort()
print("Width at height 48 (300 samples) p50/p90/p99/max:", [round(percentile(widths, p)) for p in (0.5, 0.9, 0.99)], round(widths[-1]))
print(f"Share wider than IMG_W={IMG_W}: {sum(w > IMG_W for w in widths) / len(widths):.1%} (bị nén ngang)")

dict_lines = DICT_PATH.read_text(encoding="utf-8").splitlines()
charset = set(dict_lines) | {" "}
print("Dictionary entries:", len(dict_lines), "| unique:", len(set(dict_lines)), "| has space line:", " " in dict_lines)
missing = Counter(ch for label in train_labels for ch in label if ch not in charset)
total_chars = sum(len(label) for label in train_labels)
print(f"Train characters outside dictionary: {sum(missing.values())} / {total_chars} ({sum(missing.values()) / total_chars:.4%})")
print("Most frequent missing:", missing.most_common(10))

## 8. Mã nguồn PaddleOCR và trọng số tiền huấn luyện

Huấn luyện dùng `tools/train.py` của PaddleOCR nên cell lấy mã nguồn đúng tag `v3.3.0` (cùng phiên bản với thư viện đã cài), ghi lại mã commit, cài thêm vài thư viện phụ trợ rồi tải checkpoint tiền huấn luyện `PP-OCRv5_mobile_rec_pretrained.pdparams` làm điểm khởi đầu. Lớp đầu ra có số lớp khác vì từ điển tiếng Việt, nên PaddleOCR bỏ qua các tham số không khớp kích thước và khởi tạo lại.

Cell này còn vá một hạn chế của backbone `PPLCNetV3` (`ppocr/modeling/backbones/rec_lcnetv3.py`). Hàm loss CTC đòi số bước thời gian của mô hình không nhỏ hơn độ dài nhãn. Mã gốc ép số bước này về đúng 40 khi huấn luyện, còn lúc suy luận nó là W/8 (W là chiều rộng ảnh); hai con số trùng nhau ở ảnh rộng 320 px (320/8 = 40) nên với cấu hình gốc không có vấn đề. Nhãn tiếng Việt dài trung vị 57 và tối đa 80 ký tự, nên với mã gốc khoảng 74% dòng không thể căn chỉnh và mô hình không hội tụ. Bản vá thay 40 bằng W/8 (với ảnh rộng 960 px là 120 bước, đủ cho nhãn dài nhất) và dùng chung cho cả hai lần chạy 2 và 4 epoch, nên không phải biến thí nghiệm.

In [ ]:
import urllib.request

PADDLEOCR_REF = "v3.3.0"
PADDLEOCR_DIR = Path("/kaggle/working/PaddleOCR")
if not (PADDLEOCR_DIR / "tools" / "train.py").is_file():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", PADDLEOCR_REF,
                    "https://github.com/PaddlePaddle/PaddleOCR.git", str(PADDLEOCR_DIR)], check=True)
ENV_INFO["paddleocr_source_ref"] = PADDLEOCR_REF
ENV_INFO["paddleocr_source_commit"] = subprocess.check_output(
    ["git", "-C", str(PADDLEOCR_DIR), "rev-parse", "HEAD"], text=True).strip()
print("PaddleOCR source commit:", ENV_INFO["paddleocr_source_commit"])

# Fix: training forces CTC to 40 time steps whatever the image width (inference uses W/8).
BACKBONE = PADDLEOCR_DIR / "ppocr" / "modeling" / "backbones" / "rec_lcnetv3.py"
OLD_POOL = "x = F.adaptive_avg_pool2d(x, [1, 40])"
NEW_POOL = "x = F.adaptive_avg_pool2d(x, [1, max(1, x.shape[3] // 2)])"
src = BACKBONE.read_text(encoding="utf-8")
if OLD_POOL in src:
    BACKBONE.write_text(src.replace(OLD_POOL, NEW_POOL), encoding="utf-8")
src = BACKBONE.read_text(encoding="utf-8")
assert NEW_POOL in src and OLD_POOL not in src, "rec_lcnetv3.py was not patched"
ENV_INFO["paddleocr_patch"] = "rec_lcnetv3.py: adaptive_avg_pool2d([1, 40]) -> [1, W_feat // 2] (CTC time steps = W/8 in training, as in inference)"
print("Patched:", ENV_INFO["paddleocr_patch"])

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PADDLEOCR_DIR / "requirements.txt")], check=True)
check = subprocess.run([sys.executable, "-c", "import ppocr.data, ppocr.modeling.architectures"],
                       cwd=PADDLEOCR_DIR, capture_output=True, text=True)
assert check.returncode == 0, check.stderr[-2000:]

PRETRAINED_URL = "https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv5_mobile_rec_pretrained.pdparams"
PRETRAINED_PATH = Path(config["Global"]["pretrained_model"])
PRETRAINED_PATH.parent.mkdir(parents=True, exist_ok=True)
if not PRETRAINED_PATH.is_file():
    urllib.request.urlretrieve(PRETRAINED_URL, PRETRAINED_PATH)
assert PRETRAINED_PATH.stat().st_size > 1_000_000, "Pretrained download looks truncated"
print("Pretrained:", PRETRAINED_PATH, f"({PRETRAINED_PATH.stat().st_size / 1e6:.1f} MB)")

## 9. Huấn luyện

Chạy `tools/train.py` đúng với config trong repository trên 2 GPU T4, không ghi đè tham số nào. Checkpoint được lưu sau mỗi epoch (`save_epoch_step: 1`) để còn dùng được nếu phiên Kaggle bị ngắt; PaddleOCR luôn ghi thêm `latest` (sau epoch cuối) và `best_accuracy` (tốt nhất trên tập validation `rec_val.txt`, đánh giá mỗi khoảng 1.000 bước). Báo cáo dùng `latest`, tức mô hình sau đúng 2 epoch, để so sánh 2 và 4 epoch công bằng; `best_accuracy` chỉ để tham khảo. Thời gian huấn luyện là tổng thời gian chạy lệnh (kể cả các lần đánh giá trên validation) và được đo giống nhau ở cả hai notebook.

In [ ]:
GPUS = "0,1"
OUTPUT_DIR = Path(config["Global"]["save_model_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_LOG = OUTPUT_DIR / "train_stdout.log"
TRAIN_TIME_PATH = OUTPUT_DIR / "train_time.json"
shutil.copy(CONFIG_PATH, OUTPUT_DIR / "config_used.yml")

run_env = os.environ.copy()
run_env["PYTHONPATH"] = f"{PADDLEOCR_DIR}:{run_env.get('PYTHONPATH', '')}"
run_env["PYTHONUNBUFFERED"] = "1"

train_cmd = [sys.executable, "-m", "paddle.distributed.launch", "--gpus", GPUS,
             "tools/train.py", "-c", str(CONFIG_PATH)]
print(" ".join(train_cmd))

train_start = time.perf_counter()
with TRAIN_LOG.open("w", encoding="utf-8") as log, subprocess.Popen(
        train_cmd, cwd=PADDLEOCR_DIR, env=run_env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
    for line in proc.stdout:
        log.write(line)
        print(line, end="")
    returncode = proc.wait()
train_seconds = time.perf_counter() - train_start

assert returncode == 0, f"Training failed (exit {returncode}); see {TRAIN_LOG}"
assert (OUTPUT_DIR / "latest.pdparams").is_file(), "latest.pdparams missing"
TRAIN_TIME_PATH.write_text(json.dumps(
    {"epochs": EPOCHS, "gpus": GPUS, "train_seconds": train_seconds}, indent=2) + "\n", encoding="utf-8")
print(f"Training time: {train_seconds:.1f} s ({train_seconds / 60:.1f} min)")

## 10. Xuất mô hình suy luận

Chuyển checkpoint `latest` thành mô hình suy luận (đã gắn từ điển và kích thước ảnh từ config). Mô hình này được nạp lại bằng `TextRecognition` của PaddleOCR, đúng đường dự đoán đã dùng cho baseline, nên hai bên so sánh được trực tiếp.

In [ ]:
INFER_DIR = OUTPUT_DIR / "inference"
subprocess.run(
    [sys.executable, "tools/export_model.py", "-c", str(CONFIG_PATH),
     "-o", f"Global.pretrained_model={OUTPUT_DIR / 'latest'}", f"Global.save_inference_dir={INFER_DIR}"],
    cwd=PADDLEOCR_DIR, env=run_env, check=True)
print(sorted(path.name for path in INFER_DIR.iterdir()))

## 11. Ghép nhãn với ảnh test

Giống notebook baseline: đổi đường dẫn ảnh tương đối trong `rec_test.txt` thành đường dẫn thật và kiểm tra đủ 3.003 ảnh test, để mọi số liệu được đo trên đúng tập test mà baseline đã dùng.

In [ ]:
def resolve_image_path(raw_path: str) -> Path:
    relative_path = Path(raw_path.replace("\\", "/").lstrip("./"))
    for candidate in (TEST_LABELS.parent / relative_path, DATA_DIR / relative_path):
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(f"Cannot resolve image path: {raw_path}")

test_examples = []
for line in split_lines["test"]:
    raw_path, gt = line.split("\t", 1)
    test_examples.append(
        {
            "image": raw_path.replace("\\", "/"),  # đường dẫn trong file nhãn, ghi vào JSONL
            "path": str(resolve_image_path(raw_path)),  # đường dẫn thật, dùng để dự đoán
            "gt": gt,
        }
    )

assert len(test_examples) == 3_003
assert all(Path(row["path"]).is_file() for row in test_examples)
print("Resolved test images:", len(test_examples))
print("Example:", test_examples[0])

## 12. Dự đoán toàn bộ tập test và lưu JSONL

Nạp mô hình đã fine-tune và dự đoán 3.003 ảnh test với `batch_size=1`, giống hệt baseline (batch khác nhau làm ảnh được đệm khác đi và đổi kết quả). Mỗi kết quả được ghi thành một dòng JSON gồm `image`, `gt` và `pred`, đúng định dạng mà `eval.py` và `src/jsonl.py` yêu cầu.

In [ ]:
from paddleocr import TextRecognition
from tqdm.auto import tqdm

MODEL_NAME = config["Global"]["model_name"]
BATCH_SIZE = 1

model_init_start = time.perf_counter()
recognizer = TextRecognition(model_name=MODEL_NAME, model_dir=str(INFER_DIR), device="gpu:0")
model_init_seconds = time.perf_counter() - model_init_start

def text_from_result(result) -> str:
    payload = result.json
    if isinstance(payload, str):
        payload = json.loads(payload)
    return str(payload["res"]["rec_text"])

prediction_start = time.perf_counter()
prediction_rows = []
for start in tqdm(range(0, len(test_examples), BATCH_SIZE), desc="Predicting test set"):
    batch = test_examples[start : start + BATCH_SIZE]
    results = recognizer.predict(input=[row["path"] for row in batch], batch_size=len(batch))
    assert len(results) == len(batch), "Prediction count differs from input count."
    for example, result in zip(batch, results):
        prediction_rows.append({"image": example["image"], "gt": example["gt"], "pred": text_from_result(result)})
prediction_seconds = time.perf_counter() - prediction_start
assert len(prediction_rows) == len(test_examples)

PREDICTIONS_PATH = RESULTS_DIR / f"pred_{RUN_NAME}.jsonl"
with PREDICTIONS_PATH.open("w", encoding="utf-8") as file:
    for row in prediction_rows:
        file.write(json.dumps(row, ensure_ascii=False) + "\n")

with PREDICTIONS_PATH.open(encoding="utf-8") as file:
    saved_rows = [json.loads(line) for line in file if line.strip()]
assert len(saved_rows) == len(prediction_rows)
assert all({"image", "gt", "pred"} <= row.keys() for row in saved_rows)
print(f"Prediction time: {prediction_seconds:.2f} s | Predictions: {PREDICTIONS_PATH}")
print("Sample:", prediction_rows[0])

## 13. Tính metric báo cáo

Dùng `src/metrics.py` giống baseline: `acc` và `norm_edit_dis` (giữ khoảng trắng) là số liệu chính, hai metric `*_ignore_space` là số liệu phụ. Cell cũng lưu thời gian huấn luyện, phiên bản thư viện, seed, GPU, mã commit của repository và của PaddleOCR để điền bảng so sánh (mục 2.3) và nhật ký thí nghiệm.

In [ ]:
from src.metrics import normalize_text, recognition_metrics

train_info = json.loads(TRAIN_TIME_PATH.read_text(encoding="utf-8"))
metrics = {
    "model": MODEL_NAME,
    "run": RUN_NAME,
    "split": "rec_test.txt",
    "finetune_epochs": EPOCHS,
    "checkpoint": "latest (final epoch)",
    **recognition_metrics(prediction_rows),
    "batch_size": BATCH_SIZE,
    "train_seconds": train_info["train_seconds"],
    "train_gpus": train_info["gpus"],
    "model_init_seconds": model_init_seconds,
    "prediction_seconds": prediction_seconds,
    "config": f"configs/ppocrv5_mobile/ppocrv5_mobile_{EPOCHS}epochs.yml",
    "environment": ENV_INFO,
}
METRICS_PATH = RESULTS_DIR / f"metrics_{RUN_NAME}.json"
METRICS_PATH.write_text(json.dumps(metrics, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(json.dumps(metrics, ensure_ascii=False, indent=2))

## 14. Đối chiếu bằng công cụ đánh giá của PaddleOCR (tuỳ chọn)

Chạy `tools/eval.py` của PaddleOCR trên `rec_test.txt` với checkpoint `latest` để kiểm tra chéo. `acc` của công cụ này không chuẩn hóa NFC và không dùng `src/metrics.py`, nên chỉ cần gần bằng `acc` ở trên; mục đích là xác nhận mô hình đã xuất không bị lệch so với checkpoint. Lỗi ở cell này không ảnh hưởng kết quả chính.

In [ ]:
try:
    xcheck = subprocess.run(
        [sys.executable, "tools/eval.py", "-c", str(CONFIG_PATH),
         "-o", f"Global.checkpoints={OUTPUT_DIR / 'latest'}", f"Eval.dataset.label_file_list=[{TEST_LABELS}]"],
        cwd=PADDLEOCR_DIR, env=run_env, capture_output=True, text=True, timeout=1800)
    print("\n".join((xcheck.stdout + xcheck.stderr).splitlines()[-12:]))
except Exception as exc:  # không chặn pipeline
    print("Cross-check skipped:", exc)

## 15. Lưu checkpoint lên Hugging Face và liệt kê file kết quả

Nếu có Kaggle Secret `huggingface.co`, cell đẩy thư mục `ppocrv5_mobile_2epochs/` (checkpoint `latest`, `best_accuracy`, mô hình suy luận, config, log) lên repo `nngocbao090/hcmus-vi-paddleocr-rec` để nộp làm checkpoint; trạng thái optimizer `*.pdopt` không cần cho suy luận nên bị bỏ qua. Nếu không có token thì bước này được bỏ qua và checkpoint vẫn nằm trong Output của notebook. Cuối cùng cell liệt kê hai file kết quả nhỏ (JSONL và metrics) cần đưa vào repository.

In [ ]:
HF_REPO = "nngocbao090/hcmus-vi-paddleocr-rec"
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("huggingface.co")
except Exception as exc:
    hf_token = None
    print("No huggingface.co secret, skipping upload:", exc)

if hf_token:
    from huggingface_hub import HfApi
    api = HfApi(token=hf_token)
    api.create_repo(HF_REPO, repo_type="model", exist_ok=True)
    api.upload_folder(repo_id=HF_REPO, repo_type="model", folder_path=str(OUTPUT_DIR),
                      path_in_repo=RUN_NAME, ignore_patterns=["*.pdopt"])
    print(f"Uploaded to https://huggingface.co/{HF_REPO}/tree/main/{RUN_NAME}")

print("Files to put in the repository (results/):")
for path in (PREDICTIONS_PATH, METRICS_PATH):
    print(f"{path} ({path.stat().st_size / 1024:.1f} KB)")